# SpeedrEye: entrenamiento de distancia directa

[Abrir directamente en Google Colab](https://colab.research.google.com/github/MimiKat-Fir/SpeedrEye/blob/feature/direct-geometry-distance/notebooks/train_direct_distance.ipynb)

Ejecuta las celdas en orden. En Colab se clona la rama correcta, se prepara el entorno y se guarda el modelo final en `models/distance/direct_distance.pt`. LiDAR y las etiquetas 3D de KITTI se usan solo durante el entrenamiento; la aplicacion final recibe RGB y ejecuta un unico backbone.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import random
import shutil
import subprocess
import sys
import urllib.request
import zipfile

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
REPO_URL = 'https://github.com/MimiKat-Fir/SpeedrEye.git'
BRANCH = 'feature/direct-geometry-distance'

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.95', 'tensorboard'], check=True)
    PROJECT_ROOT = Path('/content/SpeedrEye')
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)], check=True)
    os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
    os.chdir(PROJECT_ROOT)

import cv2
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox

sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline.distance.head import (
    BackboneFeatureCapture,
    DistanceRegressionHead,
    prepare_distance_inputs,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

KITTI_ROOT = PROJECT_ROOT / 'data' / 'kitti_raw'
TRAINING_DIR = KITTI_ROOT / 'training'
DETECTOR_WEIGHTS = PROJECT_ROOT / 'models' / 'yolo' / 'speedreye_best.pt'
OUTPUT_WEIGHTS = PROJECT_ROOT / 'models' / 'distance' / 'direct_distance.pt'
SMOKE_WEIGHTS = PROJECT_ROOT / 'models' / 'distance' / 'direct_distance_smoke.pt'
METRICS_PATH = PROJECT_ROOT / 'models' / 'distance' / 'direct_distance_metrics.json'
ARTIFACT_BUNDLE = PROJECT_ROOT / 'models' / 'distance' / 'direct_distance_artifacts.zip'
RUN_NAME = 'direct_distance_v1'
TENSORBOARD_ROOT = PROJECT_ROOT / 'tensorboard' / 'direct_distance'
TENSORBOARD_DIR = TENSORBOARD_ROOT / RUN_NAME

DOWNLOAD_KITTI = IN_COLAB    # Images, labels and camera calibration.
DOWNLOAD_LIDAR = False       # Optional large Velodyne archive.
RUN_SMOKE_TEST = True        # Two frames only.
RUN_FULL_TRAINING = IN_COLAB # Colab trains after the smoke test.
DOWNLOAD_ARTIFACTS = IN_COLAB
PUBLISH_TO_GITHUB = False    # Requires a GITHUB_TOKEN Colab secret.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_SIZE = 640
EPOCHS = 20
LEARNING_RATE = 1e-3
MIN_LIDAR_POINTS = 3
LIDAR_LABEL_TOLERANCE_M = 2.0
MAX_DISTANCE_M = 60.0
TARGET_CLASSES = {'Pedestrian': 0, 'Cyclist': 1}

print('Colab:', IN_COLAB)
print('Project:', PROJECT_ROOT)
print('Device:', DEVICE)
print('Dataset:', TRAINING_DIR)
print('Detector:', DETECTOR_WEIGHTS)

## 1. Dataset y objetivo

`X` es la region del objeto dentro de las caracteristicas de YOLO. `y` es la distancia frontal en metros. Se usa la mediana de puntos LiDAR proyectados dentro de la caja cuando hay suficientes puntos; si no, se usa `z` de la etiqueta 3D de KITTI. La fuente se conserva para evaluar ambos grupos por separado.

In [ ]:
CORE_ARCHIVES = {
    'data_object_image_2.zip': 'https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_image_2.zip',
    'data_object_label_2.zip': 'https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_label_2.zip',
    'data_object_calib.zip': 'https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_calib.zip',
}
LIDAR_ARCHIVE = {
    'data_object_velodyne.zip': 'https://s3.eu-central-1.amazonaws.com/avg-kitti/data_object_velodyne.zip',
}

def download_and_extract_kitti(include_lidar=False):
    KITTI_ROOT.mkdir(parents=True, exist_ok=True)
    archives = {**CORE_ARCHIVES, **(LIDAR_ARCHIVE if include_lidar else {})}
    for filename, url in archives.items():
        archive = KITTI_ROOT / filename
        if not archive.exists():
            print('Downloading:', filename)
            urllib.request.urlretrieve(url, archive)
        print('Extracting:', filename)
        with zipfile.ZipFile(archive) as compressed:
            compressed.extractall(KITTI_ROOT)

if DOWNLOAD_KITTI:
    download_and_extract_kitti(include_lidar=DOWNLOAD_LIDAR)
else:
    print('Download disabled. Place KITTI under:', KITTI_ROOT)

In [ ]:
def read_calibration(path):
    values = {}
    for line in path.read_text().splitlines():
        key, raw = line.split(':', 1)
        values[key] = np.fromstring(raw, sep=' ')
    p2 = values['P2'].reshape(3, 4)
    rectification = values['R0_rect'].reshape(3, 3)
    velo_to_camera = values['Tr_velo_to_cam'].reshape(3, 4)
    return p2, rectification, velo_to_camera

def project_lidar(path, calibration_path):
    points = np.fromfile(path, dtype=np.float32).reshape(-1, 4)
    p2, rectification, velo_to_camera = read_calibration(calibration_path)
    points_h = np.column_stack((points[:, :3], np.ones(len(points))))
    camera = (velo_to_camera @ points_h.T).T
    camera = (rectification @ camera.T).T
    valid = camera[:, 2] > 0
    camera = camera[valid]
    camera_h = np.column_stack((camera, np.ones(len(camera))))
    projected = (p2 @ camera_h.T).T
    uv = projected[:, :2] / projected[:, 2:3]
    return uv, camera[:, 2]

def parse_kitti_objects(path):
    objects = []
    for line in path.read_text().splitlines():
        fields = line.split()
        if fields[0] not in TARGET_CLASSES:
            continue
        objects.append({
            'class_name': fields[0],
            'class_id': TARGET_CLASSES[fields[0]],
            'bbox': tuple(float(value) for value in fields[4:8]),
            'kitti_z_m': float(fields[13]),
            'truncated': float(fields[1]),
            'occluded': int(fields[2]),
        })
    return objects

def lidar_distance_for_box(uv, lidar_z, bbox, expected_z):
    x1, y1, x2, y2 = bbox
    inside = (uv[:, 0] >= x1) & (uv[:, 0] <= x2) & (uv[:, 1] >= y1) & (uv[:, 1] <= y2)
    close_to_label = np.abs(lidar_z - expected_z) <= LIDAR_LABEL_TOLERANCE_M
    matched = lidar_z[inside & close_to_label]
    if len(matched) < MIN_LIDAR_POINTS:
        return np.nan, len(matched)
    return float(np.median(matched)), len(matched)

In [ ]:
image_dir = TRAINING_DIR / 'image_2'
label_dir = TRAINING_DIR / 'label_2'
calib_dir = TRAINING_DIR / 'calib'
velodyne_dir = TRAINING_DIR / 'velodyne'

if not image_dir.exists() or not label_dir.exists():
    raise FileNotFoundError('KITTI images/labels are missing. Review the previous cell.')

rows = []
for label_path in tqdm(sorted(label_dir.glob('*.txt')), desc='Building targets'):
    frame_id = label_path.stem
    image_path = image_dir / f'{frame_id}.png'
    calib_path = calib_dir / f'{frame_id}.txt'
    lidar_path = velodyne_dir / f'{frame_id}.bin'
    uv, lidar_z = (None, None)
    if lidar_path.exists() and calib_path.exists():
        uv, lidar_z = project_lidar(lidar_path, calib_path)

    for obj in parse_kitti_objects(label_path):
        lidar_distance, point_count = (np.nan, 0)
        if uv is not None:
            lidar_distance, point_count = lidar_distance_for_box(
                uv, lidar_z, obj['bbox'], obj['kitti_z_m']
            )
        distance = lidar_distance if np.isfinite(lidar_distance) else obj['kitti_z_m']
        if not 0.5 <= distance <= MAX_DISTANCE_M:
            continue
        rows.append({
            **obj,
            'frame_id': frame_id,
            'image_path': str(image_path),
            'lidar_distance_m': lidar_distance,
            'lidar_points': point_count,
            'distance_m': distance,
            'distance_source': 'lidar' if np.isfinite(lidar_distance) else 'kitti_3d',
        })

records = pd.DataFrame(rows)
if records.empty:
    raise RuntimeError('No Pedestrian/Cyclist distance targets were created.')

print('Objects:', len(records))
print(records.groupby(['class_name', 'distance_source']).size())
display(records[['class_name', 'bbox', 'distance_m', 'distance_source', 'lidar_points']].head())
display(records['distance_m'].describe())

<div style='border-left:4px solid #e0a800;padding:8px'><b>Watch out:</b> se divide por fotograma, no por objeto. Asi una imagen no puede aparecer a la vez en entrenamiento y validacion.</div>

In [ ]:
frame_ids = records['frame_id'].drop_duplicates().to_numpy()
rng = np.random.default_rng(SEED)
rng.shuffle(frame_ids)
split_index = int(0.8 * len(frame_ids))
train_frames = frame_ids[:split_index].tolist()
val_frames = frame_ids[split_index:].tolist()

train_records = records[records['frame_id'].isin(train_frames)].copy()
val_records = records[records['frame_id'].isin(val_frames)].copy()

print('Train frames/objects:', len(train_frames), len(train_records))
print('Validation frames/objects:', len(val_frames), len(val_records))

## 2. Modelo

YOLO queda congelado en esta primera version. Cada fotograma pasa una sola vez por su backbone. La nueva cabeza recibe las regiones de los objetos y aprende `log(distancia)`, que es mas estable que aprender metros directamente.

In [ ]:
if not DETECTOR_WEIGHTS.exists():
    raise FileNotFoundError(f'Missing detector weights: {DETECTOR_WEIGHTS}')

detector = YOLO(str(DETECTOR_WEIGHTS)).model.to(DEVICE).eval()
for parameter in detector.parameters():
    parameter.requires_grad = False
capture = BackboneFeatureCapture(detector)
letterbox = LetterBox(new_shape=(IMAGE_SIZE, IMAGE_SIZE), auto=False, stride=32)

def load_frame_features(frame_rows):
    image = cv2.imread(frame_rows.iloc[0]['image_path'])
    if image is None:
        raise FileNotFoundError(frame_rows.iloc[0]['image_path'])
    resized = letterbox(image=image)
    network_input = resized[:, :, ::-1].transpose(2, 0, 1).copy()
    network_input = torch.from_numpy(network_input).float().div(255).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        detector(network_input)
    feature_map = capture.features[0]
    detections = [{'bbox': bbox} for bbox in frame_rows['bbox']]
    rois, box_features = prepare_distance_inputs(
        detections, image.shape, feature_map, capture.strides[0]
    )
    target = torch.tensor(frame_rows['distance_m'].to_numpy(), device=DEVICE).float().log()
    return feature_map, rois, box_features, target

sample_rows = train_records[train_records['frame_id'] == train_frames[0]]
sample_features, sample_rois, sample_boxes, sample_target = load_frame_features(sample_rows)
head = DistanceRegressionHead(feature_channels=sample_features.shape[1]).to(DEVICE)
optimizer = torch.optim.AdamW(head.parameters(), lr=LEARNING_RATE)
loss_function = nn.SmoothL1Loss()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

DETECTOR_SHA256 = sha256_file(DETECTOR_WEIGHTS)
TENSORBOARD_DIR.mkdir(parents=True, exist_ok=True)
writer = SummaryWriter(TENSORBOARD_DIR)
writer.add_text('configuration/run_name', RUN_NAME)
writer.add_text('configuration/detector', DETECTOR_WEIGHTS.name)
writer.add_text('configuration/device', DEVICE)

def save_distance_checkpoint(path, metrics=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + '.tmp')
    torch.save({
        'format_version': 1,
        'target_mode': 'direct',
        'feature_channels': head.feature_channels,
        'hidden_channels': head.hidden_channels,
        'roi_size': head.roi_size,
        'head_state_dict': head.state_dict(),
        'detector_weights': DETECTOR_WEIGHTS.name,
        'detector_sha256': DETECTOR_SHA256,
        'image_size': IMAGE_SIZE,
        'classes': TARGET_CLASSES,
        'target': 'LiDAR median z when available, otherwise KITTI object-center z',
        'metrics': metrics or {},
    }, temporary_path)
    temporary_path.replace(path)

print('Feature map:', tuple(sample_features.shape))
print('Objects in sample:', len(sample_target))
print('Head parameters:', sum(p.numel() for p in head.parameters()))

In [ ]:
def run_epoch(frame_list, training, limit=None):
    head.train(training)
    selected = frame_list[:limit] if limit else frame_list
    losses, errors = [], []

    for frame_id in tqdm(selected, leave=False):
        frame_rows = records[records['frame_id'] == frame_id]
        feature_map, rois, box_features, y_log = load_frame_features(frame_rows)
        y_pred_log = head(feature_map, rois, box_features)
        loss = loss_function(y_pred_log, y_log)

        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        y = y_log.exp()
        y_pred = y_pred_log.detach().exp()
        losses.append(loss.item())
        errors.extend((y_pred - y).cpu().tolist())

    errors = np.asarray(errors)
    return {
        'loss': float(np.mean(losses)),
        'mae_m': float(np.mean(np.abs(errors))),
        'rmse_m': float(np.sqrt(np.mean(errors ** 2))),
    }

if RUN_SMOKE_TEST:
    smoke = run_epoch(train_frames, training=True, limit=2)
    print('Smoke test:', smoke)
    for name, value in smoke.items():
        writer.add_scalar(f'smoke/{name}', value, 0)
    writer.flush()
    save_distance_checkpoint(SMOKE_WEIGHTS, smoke)
    print('Smoke weights:', SMOKE_WEIGHTS)

In [ ]:
if RUN_FULL_TRAINING:
    best_mae = float('inf')
    for epoch in range(1, EPOCHS + 1):
        random.shuffle(train_frames)
        train_metrics = run_epoch(train_frames, training=True)
        with torch.no_grad():
            val_metrics = run_epoch(val_frames, training=False)
        print(f"Epoch {epoch:02d} | train MAE {train_metrics['mae_m']:.2f} m | val MAE {val_metrics['mae_m']:.2f} m")
        for name, value in train_metrics.items():
            writer.add_scalar(f'train/{name}', value, epoch)
        for name, value in val_metrics.items():
            writer.add_scalar(f'validation/{name}', value, epoch)

        writer.flush()
        if val_metrics['mae_m'] < best_mae:
            best_mae = val_metrics['mae_m']
            save_distance_checkpoint(OUTPUT_WEIGHTS, val_metrics)
    print('Best weights:', OUTPUT_WEIGHTS, '| validation MAE:', best_mae)
else:
    print('Full training disabled. Set RUN_FULL_TRAINING = True after reviewing the smoke test.')

In [ ]:
if RUN_FULL_TRAINING and OUTPUT_WEIGHTS.exists():
    try:
        best_checkpoint = torch.load(OUTPUT_WEIGHTS, map_location=DEVICE, weights_only=True)
    except TypeError:
        best_checkpoint = torch.load(OUTPUT_WEIGHTS, map_location=DEVICE)
    head.load_state_dict(best_checkpoint['head_state_dict'])

with torch.no_grad():
    validation = run_epoch(val_frames, training=False, limit=None if RUN_FULL_TRAINING else 2)
print('Validation:', validation)

summary = {
    'run_name': RUN_NAME,
    'detector_weights': DETECTOR_WEIGHTS.name,
    'detector_sha256': DETECTOR_SHA256,
    'train_frames': len(train_frames),
    'validation_frames': len(val_frames),
    'objects': len(records),
    'distance_sources': {
        str(source): int(count)
        for source, count in records['distance_source'].value_counts().items()
    },
    'validation': validation,
}
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
for name, value in validation.items():
    writer.add_scalar(f'final/{name}', value, EPOCHS if RUN_FULL_TRAINING else 0)
writer.flush()
writer.close()
print('Weights:', OUTPUT_WEIGHTS if OUTPUT_WEIGHTS.exists() else SMOKE_WEIGHTS)
print('Metrics:', METRICS_PATH)
print('TensorBoard:', TENSORBOARD_DIR)

## 3. Resultados y TensorBoard

Esta celda abre TensorBoard y prepara un ZIP con pesos, metricas y eventos. En Colab tambien descarga los archivos al ordenador.

In [ ]:
from IPython import get_ipython

notebook_shell = get_ipython()
if notebook_shell is not None:
    notebook_shell.run_line_magic('load_ext', 'tensorboard')
    notebook_shell.run_line_magic('tensorboard', f'--logdir "{TENSORBOARD_ROOT}"')

final_weights = OUTPUT_WEIGHTS if OUTPUT_WEIGHTS.exists() else SMOKE_WEIGHTS
with zipfile.ZipFile(ARTIFACT_BUNDLE, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for artifact in (final_weights, METRICS_PATH):
        bundle.write(artifact, artifact.relative_to(PROJECT_ROOT))
    for event_file in TENSORBOARD_DIR.rglob('*'):
        if event_file.is_file():
            bundle.write(event_file, event_file.relative_to(PROJECT_ROOT))
print('Artifact bundle:', ARTIFACT_BUNDLE)

if IN_COLAB and DOWNLOAD_ARTIFACTS:
    from google.colab import files
    files.download(str(final_weights))
    files.download(str(ARTIFACT_BUNDLE))

## 4. Compartir el modelo con el equipo

Para que todos reciban el modelo con `git pull`, crea un secreto de Colab llamado `GITHUB_TOKEN`, cambia `PUBLISH_TO_GITHUB = True` en la configuracion y ejecuta esta celda. Solo se publican los pesos finales, las metricas y este run de TensorBoard.

In [ ]:
if PUBLISH_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('The publishing helper is intended for Google Colab.')
    if not OUTPUT_WEIGHTS.exists():
        raise FileNotFoundError('Full training did not create direct_distance.pt')
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Add GITHUB_TOKEN to Colab secrets before publishing.')

    subprocess.run(['git', 'config', 'user.name', 'MimiKat-Fir'], check=True)
    subprocess.run(['git', 'config', 'user.email', '18628099+MimiKat-Fir@users.noreply.github.com'], check=True)
    subprocess.run([
        'git', 'add',
        str(OUTPUT_WEIGHTS.relative_to(PROJECT_ROOT)),
        str(METRICS_PATH.relative_to(PROJECT_ROOT)),
        str(TENSORBOARD_DIR.relative_to(PROJECT_ROOT)),
    ], check=True)
    staged_changes = subprocess.run(['git', 'diff', '--cached', '--quiet']).returncode == 1
    if staged_changes:
        subprocess.run(['git', 'commit', '-m', f'Train {RUN_NAME}'], check=True)
    authenticated_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@')
    subprocess.run(['git', 'push', authenticated_url, f'HEAD:{BRANCH}'], check=True)
    print(f'Published. Teammates can now run: git pull origin {BRANCH}')
else:
    print('Publishing disabled. The weights and ZIP were saved locally.')